# Feature Engineering From Raw FAST Data

This notebook performs the real feature-engineering step used in the FAST workflow: it starts from a raw `FAST Scores.xlsx` file plus tracking CSV files, extracts the 46 head-motion features, adds `vr_system_ordinal`, and optionally derives a modeling target such as `SUS binary`.

## 1. Configure Raw Inputs And Outputs

Set the raw FAST-style score file, the raw tracking directory, the generated output directory, and the demo config. The bundled demo uses a reduced subset under `notebook/data/raw/fast_demo` and writes the generated spreadsheets to `notebook/data/generated/headfeatures`.

In [ ]:
from pathlib import Path
import json
import sys

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebook':
    NOTEBOOK_DIR = Path('notebook')

sys.path.insert(0, str((NOTEBOOK_DIR / 'scripts').resolve()))

CONFIG_PATH = NOTEBOOK_DIR / 'configs' / 'feature_engineering_fast_sus_demo.json'
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))

# Keep paths relative to NOTEBOOK_DIR so the notebook stays GitHub-portable.
config['scores_path'] = str(NOTEBOOK_DIR / 'data' / 'raw' / 'fast_demo' / 'FAST Scores.xlsx')
config['tracking_dir'] = str(NOTEBOOK_DIR / 'data' / 'raw' / 'fast_demo' / 'Tracking')
config['output_dir'] = str(NOTEBOOK_DIR / 'data' / 'generated' / 'headfeatures')

print(CONFIG_PATH)
print(json.dumps(config, indent=2))

## 2. Inspect Raw Demo Assets

This demo subset contains a reduced `FAST Scores.xlsx` plus the matching `Tracking/` files for the selected participants. The notebook extracts the same 47 modeling variables used in the main FAST experiments.

In [ ]:
from head_metrics_schema import FEATURE_COLUMNS, FEATURE_FAMILIES
import pandas as pd

scores_preview = pd.read_excel(config['scores_path'])
tracking_files = sorted(Path(config['tracking_dir']).glob('*.csv'))

print(f"Raw score sheet: {config['scores_path']}")
print(f"Tracking directory: {config['tracking_dir']}")
print(f"Rows in raw score sheet (including the 2 Qualtrics header rows): {len(scores_preview)}")
print(f"Tracking files copied into the demo subset: {len(tracking_files)}")
display(scores_preview[['PID', 'VRSYSTEM']].iloc[2:8])

print(f'Required features: {len(FEATURE_COLUMNS)}')
for i, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f'{i:02d}. {feature}')

print('\nFeature families:')
for family, features in FEATURE_FAMILIES.items():
    print(f'- {family}: {len(features)} features')

## 3. Run Feature Engineering

This step runs the same feature-engineering logic used in the repository: it matches score rows to tracking files, extracts the 46 head-motion features, appends `vr_system_ordinal`, writes `HeadFeaturesVSSUS_BuildA/B.xlsx`, and then derives `HeadFeaturesVSSUSBinary_BuildA/B.xlsx`.

In [ ]:
from feature_engineering_pipeline import run_feature_engineering

result = run_feature_engineering(config)
manifest = result['manifest']
display(manifest)

## 4. Inspect Generated Spreadsheets

Inspect the generated `HeadFeaturesVS...` source files and the derived `SUSBinary` files that the classification notebook will use.

In [ ]:
generated_dir = Path(config['output_dir'])
headfeatures_a = pd.read_excel(generated_dir / 'HeadFeaturesVSSUS_BuildA.xlsx')
headfeatures_b = pd.read_excel(generated_dir / 'HeadFeaturesVSSUS_BuildB.xlsx')
sus_binary_a = pd.read_excel(generated_dir / 'HeadFeaturesVSSUSBinary_BuildA.xlsx')
sus_binary_b = pd.read_excel(generated_dir / 'HeadFeaturesVSSUSBinary_BuildB.xlsx')

print('HeadFeatures Build A:', headfeatures_a.shape)
print('HeadFeatures Build B:', headfeatures_b.shape)
print('SUS Binary Build A:', sus_binary_a.shape)
print('SUS Binary Build B:', sus_binary_b.shape)

display(sus_binary_a[['sus_score', 'sus_not_acceptable_target']].head())
display(sus_binary_a['sus_not_acceptable_target'].value_counts().sort_index())
display(sus_binary_b['sus_not_acceptable_target'].value_counts().sort_index())

## 5. Inspect Generated Metadata

Metadata files keep the participant-to-tracking match audit trail, including `score_pid`, `matched_tracking_pid`, `tracking_file`, and the encoded VR-system field.

In [ ]:
meta_a = pd.read_csv(generated_dir / 'HeadFeaturesVSSUS_BuildA_metadata.csv')
meta_b = pd.read_csv(generated_dir / 'HeadFeaturesVSSUSBinary_BuildB_metadata.csv')

display(meta_a.head())
display(meta_b[['score_pid', 'matched_tracking_pid', 'vr_system_ordinal', 'sus_score', 'sus_not_acceptable_target']].head())